# 00 — Environment Setup

Run this notebook first to verify the environment is configured correctly for development.

In [10]:
import sys
print(f"Python {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"

Python 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [11]:
import yaml
import openai
import dotenv
print("Core dependencies OK")

Core dependencies OK


## GPU Detection

Detects available GPU and determines how many llama.cpp layers can be offloaded.

In [12]:
def detect_compute_environment():
    result = {"cuda_available": False, "vram_gb": 0, "recommended_gpu_layers": 0}

    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            result["cuda_available"] = True
            result["vram_gb"] = round(props.total_memory / (1024 ** 3), 1)
            result["gpu_name"] = props.name
            result["recommended_gpu_layers"] = 35 if result["vram_gb"] >= 6 else 0
        else:
            result["gpu_name"] = "none"
    except ImportError:
        result["gpu_name"] = "torch not installed"

    return result

env = detect_compute_environment()
for key, val in env.items():
    print(f"{key}: {val}")

cuda_available: True
vram_gb: 8.0
recommended_gpu_layers: 35
gpu_name: NVIDIA GeForce RTX 3070 Laptop GPU


## Config System Check

In [13]:
import sys
sys.path.insert(0, '../../')

from src.utils.config_loader import load_config

config = load_config("openai")
print("OpenAI config keys:", list(config.keys()))
print("LLM model:", config["llm"]["model"])
print("Cache enabled:", config["cache"]["enabled"])

OpenAI config keys: ['llm', 'cache', 'rate_limiter']
LLM model: gpt-4o-mini
Cache enabled: True


## Environment File Check

In [14]:
import os
from pathlib import Path

env_file = Path('../../.env')

if env_file.exists():
    print(".env file found")
    api_key = os.environ.get('OPENAI_API_KEY', '')
    if api_key and not api_key.startswith('sk-your'):
        print("OPENAI_API_KEY is set")
    else:
        print("OPENAI_API_KEY not set — copy .env.example to .env and fill in values")
else:
    print(".env not found — copy .env.example to .env")

model_path = os.environ.get('LOCAL_MODEL_PATH', '')
if model_path and Path(model_path).exists():
    print(f"Local model found: {model_path}")
else:
    print("LOCAL_MODEL_PATH not set or file missing — local inference will not be available")

.env not found — copy .env.example to .env
LOCAL_MODEL_PATH not set or file missing — local inference will not be available
